# 26 — Adaptive RAG

> **Tier 6 | Ensemble & Meta**

## What is Adaptive RAG?

NB 25 (Ensemble RAG) always runs all three strategies — maximising coverage but paying
3× the latency and cost. **Adaptive RAG** inserts a **query classifier** before retrieval:
the classifier assigns each query to one strategy, so only the best-fit path runs.

### Query taxonomy and routing

| Query type | Signal | Routed to |
|-----------|--------|----------|
| `factual` | Specific entity, date, number, acronym | **Simple RAG** — fast, direct match |
| `conceptual` | How/why/explain/describe | **HyDE** — hypothesis bridges vocab gap |
| `comparative` | Compare, difference, vs, contrast | **Fusion RAG** — multiple angles needed |
| `exploratory` | Overview, what topics, what does X cover | **Fusion RAG** — broad coverage needed |

The classifier is a zero-shot LLM call with a structured output (one of the four labels).
This adds ~500ms but avoids 2–3× that in wasted strategy calls.

## PDF Framework: pdfminer.six `extract_text()`

NB 20 used `LTTextBox` layout objects; NB 21 used a heuristic element classifier.
This notebook uses the **high-level `extract_text()` function** from `pdfminer.high_level` —
the simplest pdfminer API, analogous to pypdf's `page.extract_text()` but via pdfminer's
layout engine which handles ligatures and encoding edge cases better.

| pdfminer API | Returns | Used in |
|-------------|---------|--------|
| `LTTextBox` objects | Layout boxes with y0 coordinates | NB 20 — zone tagging |
| Heuristic classifier | Element types (Title/NarrativeText) | NB 21 — recursive chunking |
| **`extract_text(path)`** | **Plain string, full document** | **NB 26 — simple flat extraction** |

## Flow Diagram


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pdfminer extract_text"]
        PDF(["📄 climate.pdf"])
        PDF --> PM["pdfminer.high_level\nextract_text()"]
        PM --> CH["Text chunks"]
        CH --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\nadaptive_rag_26")]
    end
    subgraph ADAPT ["⚡  ADAPTIVE ROUTING"]
        Q(["❓ Query"])
        Q --> CLS["LLM Classifier\nfactual / conceptual\ncomparative / exploratory"]
        CLS -->|factual| S1["Simple RAG\n~200ms"]
        CLS -->|conceptual| S2["HyDE\n~4s"]
        CLS -->|comparative\nexploratory| S3["Fusion RAG\n~2s"]
        S1 --> QDRANT
        S2 --> QDRANT
        S3 --> QDRANT
        S1 --> GEN["LLM Generation"]
        S2 --> GEN
        S3 --> GEN
        GEN --> ANS(["✅ Answer"])
    end
    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style ADAPT fill:#fdf4ff,stroke:#9333ea,color:#3b0764
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "pdfminer.six", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid
from typing import List, Dict
from collections import defaultdict
from dotenv import load_dotenv

import boto3
from pdfminer.high_level import extract_text
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

COLLECTION_NAME  = "adaptive_rag_26"
EMBEDDING_DIM    = 1024
CHUNK_SIZE       = 500
CHUNK_OVERLAP    = 50
TOP_K            = 5
N_FUSION_VARIANTS = 3
RRF_K            = 60

QUERY_TYPES = ["factual", "conceptual", "comparative", "exploratory"]

PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection   : {COLLECTION_NAME}")
print(f"PDF          : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")
print(f"Query types  : {QUERY_TYPES}")


## Step 3 — Qdrant Setup

In [ ]:
def make_qdrant(url='', api_key=''):
    if url:
        try:
            kw = {'url': url}
            if api_key: kw['api_key'] = api_key
            client = QdrantClient(**kw)
            client.get_collections()
            print(f'Qdrant Cloud: {url}')
            return client
        except Exception as e:
            print(f'Qdrant Cloud unavailable ({e}), falling back...')
    print('Using Qdrant in-memory')
    return QdrantClient(':memory:')


qdrant = make_qdrant(QDRANT_URL, QDRANT_API_KEY)

if COLLECTION_NAME in [c.name for c in qdrant.get_collections().collections]:
    qdrant.delete_collection(COLLECTION_NAME)
qdrant.create_collection(
    COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
print(f'Created "{COLLECTION_NAME}" (dim={EMBEDDING_DIM})')


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

def call_llm(prompt: str, system: str = '', max_tokens: int = 512) -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system or "You are a helpful assistant.",
        "messages": [{"role": "user", "content": prompt}],
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"]

test_emb = embed_text("adaptive rag classifier routing test")
print(f"Embedding OK — dim={len(test_emb)}")
print("call_llm ready.")


## Step 5 — PDF Loading with pdfminer.six `extract_text()`

`pdfminer.high_level.extract_text(path)` returns the full document as one string.
We split it into fixed-size overlapping chunks — no page metadata needed here
since the focus is routing accuracy, not source attribution.


In [ ]:
def recursive_split(text: str, size: int, overlap: int) -> List[str]:
    if len(text) <= size:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + size].strip()
        if chunk:
            chunks.append(chunk)
        start += size - overlap
    return chunks


def load_pdf_pdfminer(path: str):
    raw   = extract_text(path)
    # collapse excessive whitespace
    text  = ' '.join(raw.split())
    subs  = recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP)
    chunks = [{'text': s, 'char_count': len(s)} for s in subs]
    stats  = {
        'total_chars': len(text),
        'n_chunks'   : len(chunks),
        'avg_chars'  : sum(c['char_count'] for c in chunks) // max(len(chunks), 1),
    }
    return chunks, stats


t0 = time.time()
chunks, stats = load_pdf_pdfminer(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"pdfminer extract_text() : {elapsed:.0f}ms")
print(f"Total chars             : {stats['total_chars']}")
print(f"Chunks                  : {stats['n_chunks']}")
print(f"Avg chunk length        : {stats['avg_chars']} chars")


## Step 6 — Index

In [ ]:
print(f"Embedding {len(chunks)} chunks...")
t0   = time.time()
embs = embed_batch([c['text'] for c in chunks], label='[index]')

pts = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector=embs[i],
        payload={
            'text'    : chunks[i]['text'],
            'metadata': {
                'char_count': chunks[i]['char_count'],
                'source'    : 'climate.pdf',
                'pdf_lib'   : 'pdfminer-extract_text',
            }
        }
    )
    for i in range(len(chunks))
]
qdrant.upsert(collection_name=COLLECTION_NAME, points=pts)
print(f"Indexed {qdrant.get_collection(COLLECTION_NAME).points_count} vectors in {time.time()-t0:.1f}s")


## Step 7 — Query Classifier

A zero-shot LLM call returns exactly one label from `{factual, conceptual, comparative, exploratory}`.
The model is prompted to output only the label — no explanation — so it can be parsed directly.


In [ ]:
CLASSIFIER_SYSTEM = """You are a query classifier for a RAG system.
Classify the user's question into exactly one of these categories:

  factual     — asks for a specific fact, entity, date, number, or definition
  conceptual  — asks how/why something works, or to explain a concept
  comparative — asks to compare, contrast, or identify differences between things
  exploratory — asks for an overview, summary, or what topics a document covers

Output ONLY the category label, nothing else.
"""


def classify_query(question: str) -> str:
    label = call_llm(
        question,
        system=CLASSIFIER_SYSTEM,
        max_tokens=10,
    ).strip().lower()
    # guard: map to nearest valid label
    for valid in QUERY_TYPES:
        if valid in label:
            return valid
    return "factual"   # safe default


# Smoke test
test_pairs = [
    ("When was the first weather satellite launched?", "factual"),
    ("How does ensemble forecasting work?",           "conceptual"),
    ("What is the difference between NWP and climatology forecasting?", "comparative"),
    ("What topics does this document cover?",         "exploratory"),
]
print(f"{'Question':<55} {'Expected':>11} {'Got':>11}")
print("-" * 80)
for q, expected in test_pairs:
    got = classify_query(q)
    match = "✓" if got == expected else "✗"
    print(f"{q[:54]:<55} {expected:>11} {got:>11}  {match}")


## Step 8 — Retrieval Strategies

In [ ]:
def vector_search(qvec: List[float], top_k: int = TOP_K) -> List[Dict]:
    resp = qdrant.query_points(
        collection_name=COLLECTION_NAME, query=qvec,
        limit=top_k, with_payload=True)
    return [{'text': p.payload['text'],
             'metadata': p.payload.get('metadata', {}),
             'score': p.score} for p in resp.points]


def rrf_merge(result_lists: List[List[Dict]], top_k: int = TOP_K, k: int = RRF_K) -> List[Dict]:
    scores: Dict[str, float] = defaultdict(float)
    lookup: Dict[str, Dict]  = {}
    for lst in result_lists:
        for rank, item in enumerate(lst, start=1):
            key = item['text'][:120]
            scores[key] += 1.0 / (k + rank)
            if key not in lookup:
                lookup[key] = item
    return [{**lookup[k], 'rrf_score': s}
            for k, s in sorted(scores.items(), key=lambda x: -x[1])[:top_k]]


def retrieve_simple(question: str) -> List[Dict]:
    return vector_search(embed_text(question))


def retrieve_hyde(question: str) -> List[Dict]:
    hypo = call_llm(
        f"Write a short factual passage (3-4 sentences) answering: {question}",
        system="You are a technical writer. Write only the passage.",
        max_tokens=200,
    )
    return vector_search(embed_text(hypo))


def retrieve_fusion(question: str) -> List[Dict]:
    variants_raw = call_llm(
        f"Generate {N_FUSION_VARIANTS} distinct search queries for: {question}\n"
        "Output one query per line, no numbering.",
        system="You are a search query generator. Output only the queries.",
        max_tokens=200,
    )
    queries = [q.strip() for q in variants_raw.strip().splitlines() if q.strip()][:N_FUSION_VARIANTS]
    return rrf_merge([vector_search(embed_text(q)) for q in queries])


STRATEGY_MAP = {
    "factual"    : retrieve_simple,
    "conceptual" : retrieve_hyde,
    "comparative": retrieve_fusion,
    "exploratory": retrieve_fusion,
}

print("Retrieval strategies defined.")
print("Routing map:")
for qtype, fn in STRATEGY_MAP.items():
    print(f"  {qtype:<12} → {fn.__name__}")


## Step 9 — Adaptive RAG Pipeline

In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about a climate document.
Answer ONLY from the provided context. Be concise and precise.
If the answer is not in the context, say so explicitly.
"""


def adaptive_rag(question: str, verbose: bool = True) -> Dict:
    t0 = time.time()

    # 1. Classify query
    t_cls   = time.time()
    qtype   = classify_query(question)
    t_cls   = (time.time() - t_cls) * 1000

    # 2. Route to best strategy
    strategy_fn = STRATEGY_MAP[qtype]
    t_ret = time.time()
    hits  = strategy_fn(question)
    t_ret = (time.time() - t_ret) * 1000

    # 3. Generate
    context  = '\n\n'.join(
        f"[{i+1}] score={h.get('rrf_score', h.get('score', 0)):.4f}\n{h['text']}"
        for i, h in enumerate(hits)
    )
    user_msg = f"Context:\n{context}\n\nQuestion: {question}"
    t_gen    = time.time()
    answer   = call_llm(user_msg, system=SYSTEM_PROMPT, max_tokens=1024)
    t_gen    = (time.time() - t_gen) * 1000
    latency  = (time.time() - t0) * 1000

    if verbose:
        print(f"Q        : {question}")
        print(f"Type     : {qtype}  →  {strategy_fn.__name__}")
        print(f"Latency  : classify={t_cls:.0f}ms  retrieve={t_ret:.0f}ms  generate={t_gen:.0f}ms  total={latency:.0f}ms")
        print(f"A        : {answer[:500]}")
        print()

    return {
        'question'      : question,
        'query_type'    : qtype,
        'strategy'      : strategy_fn.__name__,
        'answer'        : answer,
        'n_chunks'      : len(hits),
        'latency_ms'    : latency,
        't_classify_ms' : t_cls,
        't_retrieve_ms' : t_ret,
        't_generate_ms' : t_gen,
    }


print("adaptive_rag() defined.")


## Step 10 — Demo

Eight questions designed to trigger each of the four query types,
verifying that the router selects the expected strategy.


In [ ]:
test_questions = [
    # factual
    "When was the first meteorological satellite launched?",
    "How many land-based observation stations are in the WMO network?",
    # conceptual
    "How does Numerical Weather Prediction work?",
    "Why do long-range forecasts have lower accuracy than short-range ones?",
    # comparative
    "What is the difference between persistence and climatology forecasting methods?",
    "How do satellite observations compare to ground-based ones for data collection?",
    # exploratory
    "What topics does this document cover about weather observation systems?",
    "Give me an overview of the forecasting methods described in the document.",
]

results = []
print("=" * 70)
for q in test_questions:
    r = adaptive_rag(q, verbose=True)
    results.append(r)
    print("-" * 70)


## Step 11 — Routing Analysis

In [ ]:
from collections import Counter

type_counts = Counter(r['query_type'] for r in results)
print(f"{'Query type':<14} {'Count':>6}  {'Avg classify':>13}  {'Avg retrieve':>13}  {'Avg total':>10}")
print("-" * 62)
for qtype in QUERY_TYPES:
    typed = [r for r in results if r['query_type'] == qtype]
    if not typed:
        continue
    avg_cls = sum(r['t_classify_ms'] for r in typed) / len(typed)
    avg_ret = sum(r['t_retrieve_ms'] for r in typed) / len(typed)
    avg_tot = sum(r['latency_ms']    for r in typed) / len(typed)
    print(f"{qtype:<14} {len(typed):>6}  {avg_cls:>12.0f}ms  {avg_ret:>12.0f}ms  {avg_tot:>9.0f}ms")

print()
print(f"{'Q':<4} {'Type':<13} {'Strategy':<22} {'Total':>8}  Question")
print("-" * 80)
for i, r in enumerate(results, 1):
    print(f"{i:<4} {r['query_type']:<13} {r['strategy']:<22} {r['latency_ms']:>7.0f}ms  {r['question'][:38]}")


## Step 12 — Summary

### Adaptive RAG vs Ensemble RAG

| Dimension | Ensemble RAG (NB 25) | Adaptive RAG (NB 26) |
|-----------|---------------------|---------------------|
| Strategies run | All 3 always | 1, chosen by classifier |
| Latency | ~12s (3 strategies) | ~3-6s (1 strategy + classifier) |
| Cost | 3× LLM calls | 2× LLM calls (classify + generate) |
| Coverage | Maximum | Best-fit for query type |
| Risk | Over-retrieval | Mis-classification |

### Routing decision table

| Query type | Signal words | Strategy | Why |
|-----------|-------------|----------|-----|
| factual | who/when/how many/what is X | Simple RAG | Direct match, low latency |
| conceptual | how does/why/explain/describe | HyDE | Hypothesis bridges vocab gap |
| comparative | compare/difference/vs/contrast | Fusion RAG | Multiple angles needed |
| exploratory | overview/what topics/summarise | Fusion RAG | Broad coverage needed |

### pdfminer API surface used across the series

| Notebook | pdfminer API | Purpose |
|----------|-------------|--------|
| NB 20 | `LTTextBox`, `LTPage` | Layout-zone tagging via y0 |
| NB 21 | `PDFPageInterpreter` + heuristic classifier | Element-type hierarchy |
| **NB 26** | **`extract_text(path)`** | **Flat plain-text extraction** |

> **Next: [27 — Streaming RAG](../tier7_production/27_Streaming_RAG.ipynb)** —
> stream Bedrock response tokens in real-time using the `invoke_model_with_response_stream` API.


In [ ]:
print(f"Collection '{COLLECTION_NAME}': {qdrant.get_collection(COLLECTION_NAME).points_count} vectors")
print(f"PDF framework  : pdfminer.high_level.extract_text() — flat plain-text")
print(f"RAG pattern    : Adaptive RAG — LLM classifier routes to Simple / HyDE / Fusion")
print(f"Query types    : {QUERY_TYPES}")
print()
print("Notebook 26 — Adaptive RAG complete.")
